In [ ]:
# A = [[1, 2], [3, 4]]
# B = [[1, 2, 3], [4, 5, 6]]
import math

def matrix_mult(M, N):
    m, n, r = len(M), len(M[0]), len(N[0])
    assert n == len(N), f"{m}x{n} and {len(N)}x{r} matrices sizes incompatible"

    result = [[0 for _ in range(r)] for _ in range(m)]

    for i in range(m):
        for j in range(r):
            sum = 0
            for k in range(n):
                sum += M[i][k] * N[k][j]
            result[i][j] = sum
    return result



def matrix_add(M, N):
    m, n = len(M), len(M[0])
    assert m == len(N) and n == len(N[0]), f"{m}x{n} and {len(N)}x{len(N[0])} matrices sizes incompatible"

    result = [[M[i][j] + N[i][j] for j in range(n)] for i in range(m)]
    return result

def matrix_sub(M, N):
    m, n = len(M), len(M[0])
    assert m == len(N) and n == len(N[0]), f"{m}x{n} and {len(N)}x{len(N[0])} matrices sizes incompatible"

    result = [[M[i][j] - N[i][j] for j in range(n)] for i in range(m)]
    return result

def hadamard(M, N):
    m, n = len(M), len(M[0])
    assert m == len(N) and n == len(N[0]), f"{m}x{n} and {len(N)}x{len(N[0])} matrices sizes incompatible"

    result = [[M[i][j] * N[i][j] for j in range(n)] for i in range(m)]
    return result


def transpose(M):
    m, n = len(M), len(M[0])
    # print(f"{m} x {n}")
    transposed = [[0.0 for _ in range(m)] for _ in range(n)]

    for i in range(m):
        for j in range(n):
            transposed[j][i] = M[i][j]

    return transposed


def sigmoid(x):
    return 1 / (1 + math.exp(-x))

def dsigmoid(x):
    return sigmoid(x) * (1-sigmoid(x))

def vector_dsigmoid(z):
    return [[dsigmoid(z[i][0])] for i in range(len(z))]


def vector_sigmoid(z):
    return [[sigmoid(z[i][0])] for i in range(len(z))]


def C_x(y, a):
    loss, n = 0, len(y)
    for i in range(n):
        loss += (y[i]-a[i]) ** 2
    loss /= 2
    return loss

class Layer:
    def __init__(self, input_neurons, output_neurons):
        self.j = output_neurons
        self.k = input_neurons

        self.W = [[0.0 for _ in range(self.k)] for _ in range(self.j)]
        self.b = [[0.0] for _ in range(self.j)]

        self.W_grad = [[0.0 for _ in range(self.k)] for _ in range(self.j)]
        self.b_grad = [[0.0] for _ in range(self.j)]

    def __call__(self, prev_activations):
        return self.forward(prev_activations)

    def forward(self, prev_activations):
        z = matrix_add(matrix_mult(self.W, prev_activations), self.b)
        return vector_sigmoid(z), z

class Optimizer:
    def __init__(self, lr):
        self.lr = lr

    def __call__(self, layer): # update the parameters
        layer.W = matrix_sub(layer.W, self.lr * layer.W_grad)
        layer.b = matrix_sub(layer.b, self.lr * layer.b_grad)




class MLP:
    def __init__(self, hidden_neurons):
        self.layers = [
            Layer(784, hidden_neurons),
            Layer(hidden_neurons, 10),
        ]
        self.activations = []
        self.weighted_inputs = []

        # self.weight_grads = []
        # self.bias_grads = []

        self.optim = Optimizer(1e-3)



    def __call__(self, x):
        return self.forward(x)

    def forward(self, x):
        output = x

        for layer in self.layers: 
            output, weighted_input = layer(output)
            self.activations.append(output)
            self.weighted_inputs.append(weighted_input)

        return output

    def backward(self, x, y):
        L = len(self.layers)
        delta_l_s = [0.0 for _ in range(L)]

        for l in range(L-1, -1, -1):
            # print(l)
            a_l = self.activations[l]
            z_l = self.weighted_inputs[l]

            if (l == L - 1):
                delta_l = hadamard(matrix_sub(a_l, y), vector_dsigmoid(z_l))
            else:
                delta_l = hadamard(matrix_mult(transpose(self.layers[l+1].W), delta_l_s[l+1]), vector_dsigmoid(z_l))

            delta_l_s[l] = delta_l

            if (l == 0):
                self.layers[-1].W_grad =  matrix_mult(delta_l, transpose(x)) 
            else:
                self.layers[-1].W_grad =  matrix_mult(delta_l, transpose(self.activations[l-1]))

            self.layers[-1].b_grad = delta_l


    def step(self): # update parameters 
        for i in range(len(self.layers)):
            self.optim(self.layers[i])
        


mlp = MLP(hidden_neurons=30)

x = [[1.0] for i in range(784)] 
y = [[0.0] for i in range(10)]
y[0] = [1.0]

print(mlp(x))
mlp.backward(x, y) 



         


[[0.5], [0.5], [0.5], [0.5], [0.5], [0.5], [0.5], [0.5], [0.5], [0.5]]
